In [1]:
%load_ext autoreload
%autoreload 2

#### Importing

In [2]:
import json
from functools import partial
import numpy as np
import pandas as pd
import torch
from fancy_einsum import einsum

from transformers import AutoModelForCausalLM
from datasets import load_from_disk


import transformer_lens as tl
from circuitsvis.attention import attention_heads
import transformer_lens.utils as utils
from transformer_lens import ActivationCache, HookedTransformer, HookedTransformerConfig

from geomechinterp.causal.mygpt import SymbolTokenizer, DataCollator
from geomechinterp.tflens.utils import load_gpt2_to_hooked_transformer
from geomechinterp.causal.base_functions import ALL_BINARY_GENERATOR_FUNC_NAMES
from geomechinterp.causal.utils import DisplayChain

In [3]:
from IPython.display import HTML, IFrame
from tqdm import tqdm

import plotly.express as px
import plotly.graph_objects as go
from matplotlib import pyplot as plt
import plotly.io as pio

#### Loading Model

In [4]:
model_hf = AutoModelForCausalLM.from_pretrained("../gpt_rope_custom/checkpoint-407000/")


hook_config = HookedTransformerConfig(d_vocab=29, n_ctx=128, d_model=128,
                                      d_head=32, n_layers=6, n_heads=4,
                                      rotary_dim=64, act_fn='gelu_new', 
                                      original_architecture='GPT2LMHeadModel')

model = load_gpt2_to_hooked_transformer(model_hf, hook_config)

# model.tokenizer = SymbolTokenizer()

model_hf.to("mps").eval()
model.to("mps").eval()
print('')

Moving model to device:  mps



In [6]:
from datasets import Dataset

tokenizer = SymbolTokenizer()
dataset = load_from_disk("../tokenized_train")
test_dataset = load_from_disk("../tokenized_test")

train_collator = DataCollator(dataset, device="mps")  
test_collator = DataCollator(test_dataset, device="mps")  

# select subset of deterministic patterns
deterministic_train_dataset = []
for case in dataset:
    if case['stochastic']:
        continue
    else:
        deterministic_train_dataset.append(case)

deterministic_test_dataset = []
for case in test_dataset:
    if case['stochastic']:
        continue
    else:
        deterministic_test_dataset.append(case)

print(len(deterministic_train_dataset))
print(len(deterministic_test_dataset))

deterministic_test_dataset = Dataset.from_list(deterministic_test_dataset)
deterministic_train_dataset = Dataset.from_list(deterministic_train_dataset)

# drop all rows with text length < 5
deterministic_train_dataset = deterministic_train_dataset.filter(lambda x: len(x['text']) > 5)
deterministic_test_dataset = deterministic_test_dataset.filter(lambda x: len(x['text']) > 5)

1011
46


Filter:   0%|          | 0/1011 [00:00<?, ? examples/s]

Filter:   0%|          | 0/46 [00:00<?, ? examples/s]

#### Attention Patterns

In [37]:
def logits_to_ave_logit_diff(logits, answer_tokens, per_prompt=False):
    # Only the final logits are relevant for the answer
    final_logits = logits[:, -1, :]
    answer_logits = final_logits.gather(dim=-1, index=answer_tokens)
    answer_logit_diff = answer_logits[:, 0] - answer_logits[:, 1]
    if per_prompt:
        return answer_logit_diff
    else:
        return answer_logit_diff.mean()


prompts = deterministic_train_dataset['text'][:5]
answer_tokens = [tokenizer.encode(p[10]) for p in prompts]
tokenized_prompts = [tokenizer.encode(p[:10]) for p in prompts]

input_ids = torch.tensor(tokenized_prompts).to("mps")
attention_mask = torch.ones_like(input_ids).to("mps")
answer_tokens = torch.tensor(answer_tokens).to("mps")

original_logits, cache = model.run_with_cache(input_ids, attention_mask=attention_mask)
# original_average_logit_diff = logits_to_ave_logit_diff(original_logits, answer_tokens)


IndexError: index 1 is out of bounds for dimension 1 with size 1

In [ ]:


final_logits = original_logits[:, -1, :]
answer_logits = final_logits.gather(dim=-1, index=answer_tokens)
answer_logit_diff = answer_logits[:, 0] - answer_logits[:, 1]

In [47]:
original_logits

tensor([[[  3.0995,   2.9649,   2.7675,  ..., -20.6176, -20.6228, -20.6148],
         [  1.1512,   0.6101,   0.4241,  ...,  -2.0026,  -2.0032,  -2.0083],
         [ -6.8088,  -6.9602,  -6.8738,  ..., -19.6240, -19.6331, -19.6285],
         ...,
         [ -0.4213,  -0.6402,  -0.5848,  ...,  -2.4410,  -2.4392,  -2.4492],
         [ -6.9010,  -7.0422,  -6.9495,  ..., -19.1844, -19.1924, -19.1897],
         [  3.0623,   3.0674,   2.9051,  ..., -21.8363, -21.8497, -21.8367]],

        [[  3.2503,   2.7638,   2.5235,  ..., -22.7267, -22.7367, -22.7211],
         [  1.4297,   0.8685,   0.6838,  ...,  -3.2877,  -3.2901,  -3.2948],
         [ -6.6194,  -6.7730,  -6.8769,  ..., -24.4568, -24.4735, -24.4539],
         ...,
         [  0.7171,   0.8982,   0.3970,  ...,  -0.7962,  -0.7923,  -0.8036],
         [ -6.8829,  -7.0701,  -6.9747,  ..., -21.4254, -21.4365, -21.4281],
         [  3.3539,   2.9561,   2.1715,  ..., -22.1000, -22.1075, -22.0987]],

        [[ -6.4871,  -7.0209,  -6.8272,  ...

In [42]:
answer_tokens.shape

torch.Size([5, 1])

In [40]:
answer_logits.shape

torch.Size([5, 1])

In [25]:
model.tokenizer

In [10]:
prompts

['+a +a +b -b +a +a +b -b +a +a +b -b +a +a +b -b +a +a +b -b +a +a +b -b +a +a +b -b +a +a',
 '-a -a -b -b -a -a -b -b -a -a -b -b -a -a -b -b -a -a -b -b -a -a -b -b -a -a -b -b -a -a',
 'B+ a- a- A+ B+ a- a- A+ B+ a- a- A+ B+ a- a- A+ B+ a- a- A+ B+ a- a- A+ B+ a- a- A+ B+ a-',
 'a+ a+ b- b- a+ a+ b- b- a+ a+ b- b- a+ a+ b- b- a+ a+ b- b- a+ a+ b- b- a+ a+ b- b- a+ a+',
 'A+ a- a+ b+ A+ a- a+ b+ A+ a- a+ b+ A+ a- a+ b+ A+ a- a+ b+ A+ a- a+ b+ A+ a- a+ b+ A+ a-',
 'a- a+ B+ B+ a- a+ B+ B+ a- a+ B+ B+ a- a+ B+ B+ a- a+ B+ B+ a- a+ B+ B+ a- a+ B+ B+ a- a+',
 'a+ a+ A+ a+ A+ a+ A+ a+ A+ a+ A+ a+ A+ a+ A+ a+ A+ a+ A+ a+ A+ a+ A+ a+ A+ a+ A+ a+ A+ a+',
 'A- a+ a+ a- a+ a- a+ a- a+ a- a+ a- a+ a- a+ a- a+ a- a+ a- a+ a- a+ a- a+ a- a+ a- a+ a-',
 'B A a b B A a b B A a b B A a b B A a b B A a b B A a b B A',
 'B- a- a+ A+ B- a- a+ A+ B- a- a+ A+ B- a- a+ A+ B- a- a+ A+ B- a- a+ A+ B- a- a+ A+ B- a-']

In [20]:
tokenizer = SymbolTokenizer()

In [15]:
prompts[0]

'+a +a +b -b +a +a +b -b +a +a +b -b +a +a +b -b +a +a +b -b +a +a +b -b +a +a +b -b +a +a'

#### 